# Module 02 — Environment Design

Explore tasks, models, and how the Environment orchestrates them.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../../src'))
from openenv_env.tasks import TASK_REGISTRY, DEFAULT_TASK_ORDER
from openenv_env.models import Action, Observation
from openenv_env import CodeDebugEnvironment

## Explore the Task Registry

In [ ]:
print(f"Total tasks: {len(TASK_REGISTRY)}")
for tid, task in TASK_REGISTRY.items():
    print(f"  {tid:25s}  difficulty={task.difficulty:6s}  error_type={task.error_type}")

## Inspect a TaskSpec

In [ ]:
task = TASK_REGISTRY["task_logic_001"]
print("Description:", task.description)
print("\nBuggy code:\n", task.buggy_code)
print("\nFixed code:\n", task.fixed_code)
print("\nTest cases:")
for tc in task.test_inputs:
    print("  args:", tc["args"], "  expected:", tc["expected"])

## Validate Pydantic Models

In [ ]:
from pydantic import ValidationError
# Valid action
act = Action(fixed_code="def f(): pass", confidence=0.8)
print("Valid action:", act.model_dump())

# Invalid confidence (out of bounds)
try:
    bad = Action(fixed_code="x", confidence=1.5)
except ValidationError as e:
    print("\nValidation caught:", e.errors()[0]["msg"])

## Run a Full Episode Manually

In [ ]:
env = CodeDebugEnvironment(random_seed=0)
obs = env.reset(task_id="task_runtime_001")

print("TASK:", obs.task_id)
print("CODE:", obs.buggy_code)

fix = Action(
    fixed_code="def safe_average(numbers):\n    if not numbers:\n        return None\n    return sum(numbers) / len(numbers)\n",
    explanation="Guard against empty list with early return None.",
    confidence=0.9
)
result = env.step(fix)
print("\nreward:", result.reward)
print("solved:", result.reward >= 0.9)

## Design Exercise

Write a new `TaskSpec` below for a `KeyError` bug:

```python
buggy_code = """d = {'a': 1}\nprint(d['b'])  # KeyError"""
fixed_code  = """d = {'a': 1}\nprint(d.get('b', 'not found'))"""
```

Add it to `TASK_REGISTRY` and run an episode on it.